In [1]:
# STEP 1 (v4): More realistic fraud patterns - some overlap with genuine behavior
import pandas as pd
import numpy as np
import random
import string
import time

np.random.seed(42)
random.seed(42)

N = 20000

def rand_id(prefix, length=14):
    chars = string.ascii_letters + string.digits
    return f"{prefix}_" + "".join(random.choices(chars, k=length))

def fake_email():
    first = "".join(random.choices(string.ascii_lowercase, k=random.randint(4, 8)))
    domains = ["gmail.com", "yahoo.com", "outlook.com", "rediffmail.com"]
    return f"{first}{random.randint(1,999)}@{random.choice(domains)}"

def fake_phone():
    return "+91" + "".join(random.choices(string.digits, k=10))

methods = ["card", "netbanking", "wallet", "upi", "emi"]
method_weights = [0.42, 0.13, 0.10, 0.30, 0.05]

error_map = {
    "card": [
        ("BAD_REQUEST_ERROR", "payment_authentication", "customer", "incorrect_otp", "Payment processing failed because of incorrect OTP"),
        ("BAD_REQUEST_ERROR", "payment_authorization", "customer", "card_declined", "Card declined by the issuing bank"),
        ("GATEWAY_ERROR", "payment_authorization", "bank", "issuer_unavailable", "Card issuer is currently unavailable"),
        ("BAD_REQUEST_ERROR", "payment_authorization", "customer", "insufficient_funds", "Insufficient funds in the account"),
    ],
    "netbanking": [
        ("GATEWAY_ERROR", "payment_authorization", "bank", "bank_technical_error", "Bank server did not respond in time"),
        ("BAD_REQUEST_ERROR", "payment_authentication", "customer", "session_timed_out", "Customer took too long to authenticate"),
    ],
    "wallet": [
        ("BAD_REQUEST_ERROR", "payment_authorization", "customer", "insufficient_balance", "Insufficient balance in wallet"),
        ("GATEWAY_ERROR", "payment_processing", "gateway", "wallet_service_down", "Wallet service temporarily unavailable"),
    ],
    "upi": [
        ("BAD_REQUEST_ERROR", "payment_authorization", "customer", "payment_declined", "UPI payment declined by customer"),
        ("GATEWAY_ERROR", "payment_authorization", "bank", "upi_collect_expired", "UPI collect request expired before approval"),
        ("BAD_REQUEST_ERROR", "payment_authentication", "customer", "incorrect_pin", "Incorrect UPI PIN entered"),
    ],
    "emi": [
        ("BAD_REQUEST_ERROR", "payment_authorization", "customer", "emi_not_supported", "EMI not supported on this card"),
        ("GATEWAY_ERROR", "payment_authorization", "bank", "issuer_unavailable", "Card issuer is currently unavailable"),
    ],
}

base_customers = [f"cust_{i}" for i in range(1, 4001)]
base_devices = [f"device_{i}" for i in range(1, 3000)]
fraud_device_pool = base_devices[:25]        # small reused pool -> simulates a fraud ring
shared_home_devices = base_devices[25:75]    # 50 devices legitimately shared by a family/office

rows = []
now = int(time.time())

for i in range(N):
    is_fraud = np.random.choice([0, 1], p=[0.975, 0.025])
    method = np.random.choice(methods, p=method_weights)
    amount_inr = round(np.random.exponential(1800) + 50, 2)
    amount_paise = int(amount_inr * 100)
    created_at = now - random.randint(0, 30 * 24 * 3600)

    if is_fraud == 1:
        # usually reuses the fraud device pool, but sometimes a fresh device (harder to catch)
        device_id = random.choice(fraud_device_pool) if random.random() < 0.7 else random.choice(base_devices)
        customer_id = random.choice(base_customers)
        attempts_last_hour = np.random.randint(3, 20)
        international = np.random.choice([True, False], p=[0.35, 0.65])
        status = np.random.choice(["captured", "failed"], p=[0.35, 0.65])
    else:
        if random.random() < 0.05:
            device_id = random.choice(shared_home_devices)   # legit sharing, looks a bit suspicious but isn't
        else:
            device_id = random.choice(base_devices)
        customer_id = random.choice(base_customers)
        attempts_last_hour = np.random.randint(1, 5)
        international = np.random.choice([True, False], p=[0.03, 0.97])
        status = np.random.choice(["captured", "failed", "created"], p=[0.80, 0.14, 0.06])

    captured = status == "captured"
    fee = int(amount_paise * 0.02) if captured else 0
    tax = int(fee * 0.18) if captured else 0

    if status == "failed":
        error_code, error_step, error_source, error_reason, error_description = random.choice(error_map[method])
    else:
        error_code = error_step = error_source = error_reason = error_description = None

    rows.append({
        "id": rand_id("pay"), "entity": "payment", "order_id": rand_id("order"),
        "amount": amount_paise, "currency": "INR", "status": status, "method": method,
        "international": international, "captured": captured, "amount_refunded": 0,
        "refund_status": None, "fee": fee, "tax": tax, "email": fake_email(), "contact": fake_phone(),
        "error_code": error_code, "error_description": error_description, "error_source": error_source,
        "error_step": error_step, "error_reason": error_reason, "created_at": created_at,
        "customer_id": customer_id, "device_id": device_id,
        "attempts_last_hour": attempts_last_hour, "is_fraud": is_fraud,
    })

df = pd.DataFrame(rows)
df.to_csv("razorpay_style_transactions.csv", index=False)
print(df.shape)
print(df["is_fraud"].value_counts())

(20000, 25)
is_fraud
0    19483
1      517
Name: count, dtype: int64


In [2]:
from google.colab import files
files.download('razorpay_style_transactions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# STEP 2, Part 1: First attempt at teaching the Guard
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv("razorpay_style_transactions.csv")

# Feature 1: how many DIFFERENT customers used this same device? (fraud-ring signal)
df["device_customer_count"] = df.groupby("device_id")["customer_id"].transform("nunique")

# Feature 2: turn "method" (card/upi/etc) into separate 0/1 columns
method_dummies = pd.get_dummies(df["method"], prefix="method")

# Build our final feature table (the "clues" the Guard learns from)
features = pd.concat([
    df[["amount", "international", "attempts_last_hour", "device_customer_count"]],
    method_dummies
], axis=1)

target = df["is_fraud"]   # the "answer key"

# Split into practice set (80%) and test set (20%)
# stratify=target keeps the same ~2.5% fraud ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

print("Practice set:", X_train.shape, "| Fraud cases in it:", y_train.sum())
print("Test set:", X_test.shape, "| Fraud cases in it:", y_test.sum())

# Scale the numbers so the model doesn't get confused by different number ranges
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train a simple first model
model = LogisticRegression(class_weight="balanced", random_state=42)
model.fit(X_train_scaled, y_train)

preds = model.predict(X_test_scaled)

print("\n--- Baseline Logistic Regression results ---")
print("Precision:", round(precision_score(y_test, preds), 3))
print("Recall:", round(recall_score(y_test, preds), 3))
print("F1:", round(f1_score(y_test, preds), 3))
print("\nConfusion matrix:\n", confusion_matrix(y_test, preds))

Practice set: (16000, 9) | Fraud cases in it: 414
Test set: (4000, 9) | Fraud cases in it: 103

--- Baseline Logistic Regression results ---
Precision: 0.422
Recall: 0.913
F1: 0.577

Confusion matrix:
 [[3768  129]
 [   9   94]]


In [4]:
# STEP 2, Part 2: Upgrading to XGBoost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv("razorpay_style_transactions.csv")
df["device_customer_count"] = df.groupby("device_id")["customer_id"].transform("nunique")
method_dummies = pd.get_dummies(df["method"], prefix="method")
features = pd.concat([
    df[["amount", "international", "attempts_last_hour", "device_customer_count"]],
    method_dummies
], axis=1)
target = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

# tell XGBoost "fraud is rare (~1 in 38), pay extra attention to catching it"
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)
xgb_model.fit(X_train, y_train)

preds = xgb_model.predict(X_test)
probs = xgb_model.predict_proba(X_test)[:, 1]   # fraud probability score, not just yes/no - we'll use this later

print("--- XGBoost results ---")
print("Precision:", round(precision_score(y_test, preds), 3))
print("Recall:", round(recall_score(y_test, preds), 3))
print("F1:", round(f1_score(y_test, preds), 3))
print("\nConfusion matrix:\n", confusion_matrix(y_test, preds))

# which clues mattered most to the Guard's decisions?
importances = pd.Series(xgb_model.feature_importances_, index=features.columns).sort_values(ascending=False)
print("\nWhat the Guard paid attention to:\n", importances)

--- XGBoost results ---
Precision: 0.803
Recall: 0.951
F1: 0.871

Confusion matrix:
 [[3873   24]
 [   5   98]]

What the Guard paid attention to:
 attempts_last_hour       0.767913
device_customer_count    0.077594
international            0.050318
method_emi               0.020902
method_wallet            0.019159
method_upi               0.018021
method_card              0.015914
amount                   0.015364
method_netbanking        0.014816
dtype: float32


In [5]:
# STEP 2, Part 3: Teaching the Guard to explain itself (SHAP)
!pip install shap -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import shap

df = pd.read_csv("razorpay_style_transactions.csv")
df["device_customer_count"] = df.groupby("device_id")["customer_id"].transform("nunique")
method_dummies = pd.get_dummies(df["method"], prefix="method")
features = pd.concat([
    df[["amount", "international", "attempts_last_hour", "device_customer_count"]],
    method_dummies
], axis=1)
target = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42
)
xgb_model.fit(X_train, y_train)

# Build the "explainer" - this is what reads the model's mind
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Pick one transaction the Guard flagged as fraud, and explain WHY
preds = xgb_model.predict(X_test)
fraud_positions = np.where(preds == 1)[0]
pick = fraud_positions[0]   # just grabbing the first flagged one as an example

row = X_test.iloc[pick]
row_shap = shap_values[pick]

print("The transaction we're explaining:")
print(row)

print("\nWhy the Guard flagged it (positive = pushed toward FRAUD, negative = pushed toward GENUINE):")
contrib = pd.Series(row_shap, index=features.columns).sort_values(key=abs, ascending=False)
print(contrib)

print("\nFinal fraud probability:", xgb_model.predict_proba(X_test)[pick, 1])

The transaction we're explaining:
amount                   198675
international             False
attempts_last_hour           19
device_customer_count        21
method_card               False
method_emi                False
method_netbanking         False
method_upi                 True
method_wallet             False
Name: 2680, dtype: object

Why the Guard flagged it (positive = pushed toward FRAUD, negative = pushed toward GENUINE):
attempts_last_hour       10.336857
device_customer_count     0.880899
amount                   -0.326454
method_upi                0.230168
international            -0.125742
method_card              -0.069910
method_wallet             0.030469
method_netbanking         0.026998
method_emi               -0.005830
dtype: float32

Final fraud probability: 0.9999826


In [6]:
# STEP 2, Part 4: Picking a money-smart threshold (final piece of the Guard)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

df = pd.read_csv("razorpay_style_transactions.csv")
df["device_customer_count"] = df.groupby("device_id")["customer_id"].transform("nunique")
method_dummies = pd.get_dummies(df["method"], prefix="method")
features = pd.concat([
    df[["amount", "international", "attempts_last_hour", "device_customer_count"]],
    method_dummies
], axis=1)
target = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)
amounts_test = df.loc[X_test.index, "amount"] / 100   # convert paise -> rupees

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42
)
xgb_model.fit(X_train, y_train)
probs = xgb_model.predict_proba(X_test)[:, 1]   # fraud probability for each test transaction

FALSE_POSITIVE_COST = 50   # assumed cost (support/friction) of wrongly blocking a genuine customer
# false negative cost = the actual rupee amount of the fraud we missed

results = []
for threshold in np.arange(0.05, 0.96, 0.05):
    preds_t = (probs >= threshold).astype(int)
    false_positives = (preds_t == 1) & (y_test.values == 0)
    false_negatives = (preds_t == 0) & (y_test.values == 1)

    fp_cost = false_positives.sum() * FALSE_POSITIVE_COST
    fn_cost = amounts_test.values[false_negatives].sum()
    total_cost = fp_cost + fn_cost

    results.append({"threshold": round(threshold, 2), "false_positives": false_positives.sum(),
                     "false_negatives": false_negatives.sum(), "total_cost_rs": round(total_cost, 0)})

results_df = pd.DataFrame(results)
print(results_df)

best_row = results_df.loc[results_df["total_cost_rs"].idxmin()]
print("\nBest threshold by cost:", best_row["threshold"], "| Total cost: Rs", best_row["total_cost_rs"])

default_cost = results_df[results_df["threshold"] == 0.50]["total_cost_rs"].values[0]
print("Default 0.5 threshold cost: Rs", default_cost)
print("Money saved by tuning: Rs", round(default_cost - best_row["total_cost_rs"], 0))

    threshold  false_positives  false_negatives  total_cost_rs
0        0.05              182                3        14686.0
1        0.10              115                3        11336.0
2        0.15               90                4        10427.0
3        0.20               73                4         9577.0
4        0.25               64                4         9127.0
5        0.30               51                4         8477.0
6        0.35               35                4         7677.0
7        0.40               33                5         8310.0
8        0.45               27                5         8010.0
9        0.50               24                5         7860.0
10       0.55               20                5         7660.0
11       0.60               16                5         7460.0
12       0.65               12                5         7260.0
13       0.70                9                5         7110.0
14       0.75                6                6        

In [7]:
# STEP 3, Part 1: Teaching the Helper to diagnose and fix failures
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.read_csv("razorpay_style_transactions.csv")

# The Helper only deals with GENUINE failed payments (Guard already filtered fraud out)
# For now we use the real is_fraud label - in Step 4 we'll connect this to the
# Guard's actual predictions instead of this "answer key" shortcut.
genuine_failed = df[(df["status"] == "failed") & (df["is_fraud"] == 0)].copy()

print("Genuine failed payments needing recovery:", len(genuine_failed))
print("Total amount at risk (Rs):", round(genuine_failed["amount"].sum() / 100, 2))
print("\nBreakdown by failure reason:")
print(genuine_failed["error_reason"].value_counts())

# Map each failure reason -> the right recovery action
recovery_action_map = {
    "insufficient_funds":      "retry_later_24h",      # wait, maybe salary/payday hasn't hit yet
    "card_declined":           "send_reminder",          # ask customer to try a different card
    "incorrect_otp":           "send_reminder",          # ask them to retry carefully
    "expired_card":            "send_reminder",          # needs a new card, can't auto-retry
    "bank_technical_error":    "retry_immediately",      # transient issue, worth an instant retry
    "session_timed_out":       "send_reminder",
    "insufficient_balance":    "send_reminder",          # wallet - ask them to top up
    "wallet_service_down":     "retry_later_2h",         # transient outage, retry after a short wait
    "payment_declined":        "send_reminder",
    "upi_collect_expired":     "resend_collect_request",
    "incorrect_pin":           "send_reminder",
    "emi_not_supported":       "suggest_alt_method",
    "issuer_unavailable":      "retry_later_2h",         # bank-side outage, auto retry later
}

genuine_failed["recovery_action"] = genuine_failed["error_reason"].map(recovery_action_map)
genuine_failed["recovery_action"] = genuine_failed["recovery_action"].fillna("send_reminder")

print("\nRecovery actions assigned:")
print(genuine_failed["recovery_action"].value_counts())

Genuine failed payments needing recovery: 2740
Total amount at risk (Rs): 5080135.18

Breakdown by failure reason:
error_reason
issuer_unavailable      351
payment_declined        303
insufficient_funds      290
incorrect_otp           289
upi_collect_expired     281
card_declined           275
incorrect_pin           258
session_timed_out       190
bank_technical_error    169
wallet_service_down     137
insufficient_balance    129
emi_not_supported        68
Name: count, dtype: int64

Recovery actions assigned:
recovery_action
send_reminder             1444
retry_later_2h             488
retry_later_24h            290
resend_collect_request     281
retry_immediately          169
suggest_alt_method          68
Name: count, dtype: int64


In [8]:
# STEP 3, Part 2: Simulating whether each recovery action worked
success_rate_map = {
    "retry_immediately":      0.65,   # transient glitch, likely resolves fast
    "retry_later_2h":         0.55,   # short-term outage, decent odds
    "retry_later_24h":        0.45,   # insufficient funds - some still won't have funds by tomorrow
    "resend_collect_request": 0.50,   # UPI collect requests often get ignored
    "send_reminder":          0.30,   # needs customer action, lower response rate
    "suggest_alt_method":     0.40,   # customer has to actively switch payment method
}

genuine_failed["success_prob"] = genuine_failed["recovery_action"].map(success_rate_map)
genuine_failed["recovered"] = np.random.rand(len(genuine_failed)) < genuine_failed["success_prob"]

recovered_df = genuine_failed[genuine_failed["recovered"]]
total_at_risk = genuine_failed["amount"].sum() / 100
total_recovered = recovered_df["amount"].sum() / 100

print("Total at risk: Rs", round(total_at_risk, 2))
print("Total recovered: Rs", round(total_recovered, 2))
print("Recovery rate:", round(len(recovered_df)/len(genuine_failed)*100, 1), "%")

Total at risk: Rs 5080135.18
Total recovered: Rs 2082407.81
Recovery rate: 41.3 %


In [13]:
# STEP 3, Part 3: Helper uses Gemini (free) as the decision-making agent
!pip install google-genai -q

from google import genai
from google.genai import types
import json
import time
from getpass import getpass

api_key = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=api_key)

ALLOWED_ACTIONS = {
    "retry_immediately", "retry_later_2h", "retry_later_24h",
    "resend_collect_request", "send_reminder", "suggest_alt_method"
}

SYSTEM_PROMPT = """You are a payment recovery agent for Razorpay. Given details about a
failed payment, decide the best recovery action. You MUST pick exactly one action from
this list: retry_immediately, retry_later_2h, retry_later_24h, resend_collect_request,
send_reminder, suggest_alt_method.

Respond with ONLY valid JSON in this exact format, nothing else:
{"recovery_action": "...", "confidence": "high/medium/low", "reasoning": "one sentence why"}
"""

def get_llm_decision(transaction, max_retries=3):
    user_message = f"""Failed payment details:
- Amount: Rs {transaction['amount']/100:.2f}
- Payment method: {transaction['method']}
- Failure reason: {transaction['error_reason']}
- Error description: {transaction['error_description']}
- International: {transaction['international']}
- Attempts in last hour: {transaction['attempts_last_hour']}

What recovery action should we take?"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=user_message,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    response_mime_type="application/json",
                )
            )
            data = json.loads(response.text.strip())
            if data.get("recovery_action") not in ALLOWED_ACTIONS:
                return {"recovery_action": "send_reminder", "reasoning": "Fallback - action outside approved list", "confidence": "low"}
            return data

        except Exception as e:
            wait_time = 2 ** attempt
            print(f"  API hiccup (attempt {attempt+1}/{max_retries}): {type(e).__name__}. Waiting {wait_time}s...")
            time.sleep(wait_time)

    print("  Gemini unavailable after retries -> using rule-based fallback")
    return {
        "recovery_action": transaction["recovery_action"],
        "confidence": "low",
        "reasoning": "LLM unavailable after retries, used rule-based fallback"
    }

# Test on 10 transactions first
sample = genuine_failed.head(10)

for idx, row in sample.iterrows():
    decision = get_llm_decision(row)
    print(f"\nTxn {row['id']} | Rs {row['amount']/100:.2f} | Reason: {row['error_reason']}")
    print(f"  Rule-based said: {row['recovery_action']}")
    print(f"  LLM said: {decision['recovery_action']} (confidence: {decision.get('confidence')})")
    print(f"  Why: {decision.get('reasoning')}")

Paste your Gemini API key: ··········

Txn pay_h8l7qgATtLFxJp | Rs 1744.88 | Reason: payment_declined
  Rule-based said: send_reminder
  LLM said: suggest_alt_method (confidence: high)
  Why: Since the customer explicitly declined the UPI payment four times in the last hour, suggesting an alternative payment method is the best approach to complete the transaction.

Txn pay_fA1lquCu2r6AZD | Rs 442.64 | Reason: incorrect_otp
  Rule-based said: send_reminder
  LLM said: retry_immediately (confidence: high)
  Why: The payment failed due to an incorrect OTP, so asking the user to retry immediately with the correct OTP is the most effective action.

Txn pay_WFgF6cW1GC7dDy | Rs 1019.71 | Reason: insufficient_funds
  Rule-based said: retry_later_24h
  LLM said: suggest_alt_method (confidence: high)
  Why: The payment failed due to insufficient funds after multiple attempts, so offering an alternative payment method allows the customer to complete the transaction using another account or paymen

In [14]:
# STEP 3, Part 4: Stopping rules + audit trail (final piece of the Helper)
import pandas as pd
import numpy as np

np.random.seed(42)

MAX_ATTEMPTS_BEFORE_ESCALATION = 4   # our hard safety limit

audit_log = []

for idx, row in genuine_failed.iterrows():
    timestamp = pd.Timestamp.now().isoformat()

    if row["attempts_last_hour"] >= MAX_ATTEMPTS_BEFORE_ESCALATION:
        # STOPPING RULE: too many attempts already - don't auto-retry again, hand to a human
        entry = {
            "transaction_id": row["id"], "amount_rs": round(row["amount"]/100, 2),
            "error_reason": row["error_reason"], "attempts_last_hour": row["attempts_last_hour"],
            "decision_type": "escalated_manual_review", "recovery_action": "escalate_to_human",
            "source": "stopping_rule", "recovered": False,
            "reasoning": f"Stopping rule triggered: {row['attempts_last_hour']} attempts already made, avoiding further automated contact",
            "timestamp": timestamp,
        }
    else:
        action = recovery_action_map.get(row["error_reason"], "send_reminder")
        success_prob = success_rate_map[action]
        recovered = np.random.rand() < success_prob
        entry = {
            "transaction_id": row["id"], "amount_rs": round(row["amount"]/100, 2),
            "error_reason": row["error_reason"], "attempts_last_hour": row["attempts_last_hour"],
            "decision_type": "auto_recovery_attempted", "recovery_action": action,
            "source": "rule_based", "recovered": recovered,
            "reasoning": f"Mapped {row['error_reason']} -> {action} (assumed success rate {success_prob*100:.0f}%)",
            "timestamp": timestamp,
        }
    audit_log.append(entry)

audit_df = pd.DataFrame(audit_log)
audit_df.to_csv("recovery_audit_trail.csv", index=False)

print("Decision breakdown:")
print(audit_df["decision_type"].value_counts())

total_at_risk = audit_df["amount_rs"].sum()
total_recovered = audit_df[audit_df["recovered"]]["amount_rs"].sum()
print(f"\nTotal at risk: Rs {total_at_risk:,.2f}")
print(f"Total recovered: Rs {total_recovered:,.2f}")
print(f"Recovery rate (of total): {total_recovered/total_at_risk*100:.1f}%")

Decision breakdown:
decision_type
auto_recovery_attempted    2077
escalated_manual_review     663
Name: count, dtype: int64

Total at risk: Rs 5,080,135.18
Total recovered: Rs 1,478,166.12
Recovery rate (of total): 29.1%


In [15]:
# STEP 4: Connect Guard + Helper into ONE real pipeline
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

np.random.seed(42)

df = pd.read_csv("razorpay_style_transactions.csv")

# ---- Rebuild the Guard exactly as in Step 2 ----
df["device_customer_count"] = df.groupby("device_id")["customer_id"].transform("nunique")
method_dummies = pd.get_dummies(df["method"], prefix="method")
features = pd.concat([
    df[["amount", "international", "attempts_last_hour", "device_customer_count"]],
    method_dummies
], axis=1)
target = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
guard_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42
)
guard_model.fit(X_train, y_train)

BEST_THRESHOLD = 0.70  # from Step 2's cost-tuning

# ---- Get Guard's predictions on the TEST set only (data it's never seen) ----
test_indices = X_test.index
test_probs = guard_model.predict_proba(X_test)[:, 1]
guard_says_fraud = test_probs >= BEST_THRESHOLD

pipeline_df = df.loc[test_indices].copy()
pipeline_df["guard_fraud_prob"] = test_probs
pipeline_df["guard_predicts_fraud"] = guard_says_fraud

# only failed transactions are candidates for recovery
failed_in_test = pipeline_df[pipeline_df["status"] == "failed"].copy()
print("Failed transactions in held-out test set:", len(failed_in_test))
print("\nGuard's verdict on these (predicted):")
print(failed_in_test["guard_predicts_fraud"].value_counts())

print("\nCompare to REAL truth (only for OUR evaluation - the Helper never sees this):")
print(pd.crosstab(failed_in_test["guard_predicts_fraud"], failed_in_test["is_fraud"],
                   rownames=["Guard says fraud"], colnames=["Actually fraud"]))

# ---- Split based on the GUARD's prediction, NOT the real answer ----
blocked_by_guard = failed_in_test[failed_in_test["guard_predicts_fraud"] == True]
sent_to_helper = failed_in_test[failed_in_test["guard_predicts_fraud"] == False]

print(f"\nBlocked by Guard: {len(blocked_by_guard)} txns, Rs {blocked_by_guard['amount'].sum()/100:,.2f}")
print(f"Sent to Helper: {len(sent_to_helper)} txns, Rs {sent_to_helper['amount'].sum()/100:,.2f}")

wrongly_blocked = blocked_by_guard[blocked_by_guard["is_fraud"] == 0]
print(f"\nWrongly blocked (actually genuine): {len(wrongly_blocked)}")

fraud_slipped_through = sent_to_helper[sent_to_helper["is_fraud"] == 1]
print(f"Fraud that slipped through to Helper: {len(fraud_slipped_through)}")

# ---- Now run the Helper (rule-based + stopping rule) on what the GUARD sent it ----
recovery_action_map = {
    "insufficient_funds": "retry_later_24h", "card_declined": "send_reminder",
    "incorrect_otp": "send_reminder", "expired_card": "send_reminder",
    "bank_technical_error": "retry_immediately", "session_timed_out": "send_reminder",
    "insufficient_balance": "send_reminder", "wallet_service_down": "retry_later_2h",
    "payment_declined": "send_reminder", "upi_collect_expired": "resend_collect_request",
    "incorrect_pin": "send_reminder", "emi_not_supported": "suggest_alt_method",
    "issuer_unavailable": "retry_later_2h",
}
success_rate_map = {
    "retry_immediately": 0.65, "retry_later_2h": 0.55, "retry_later_24h": 0.45,
    "resend_collect_request": 0.50, "send_reminder": 0.30, "suggest_alt_method": 0.40,
}
MAX_ATTEMPTS_BEFORE_ESCALATION = 4

pipeline_log = []
for idx, row in sent_to_helper.iterrows():
    if row["attempts_last_hour"] >= MAX_ATTEMPTS_BEFORE_ESCALATION:
        entry = {"transaction_id": row["id"], "amount_rs": row["amount"]/100,
                  "decision_type": "escalated_manual_review", "recovered": False,
                  "actually_fraud": row["is_fraud"]}
    else:
        action = recovery_action_map.get(row["error_reason"], "send_reminder")
        success_prob = success_rate_map[action]
        recovered = np.random.rand() < success_prob
        entry = {"transaction_id": row["id"], "amount_rs": row["amount"]/100,
                  "decision_type": "auto_recovery_attempted", "recovered": recovered,
                  "actually_fraud": row["is_fraud"]}
    pipeline_log.append(entry)

pipeline_result = pd.DataFrame(pipeline_log)
pipeline_result.to_csv("pipeline_result.csv", index=False)

print("\n\n========== FULL PIPELINE FINAL REPORT (held-out test batch) ==========")
total_failed_amount = failed_in_test["amount"].sum() / 100
blocked_amount = blocked_by_guard["amount"].sum() / 100
recovered_amount = pipeline_result[pipeline_result["recovered"]]["amount_rs"].sum()
escalated_amount = pipeline_result[pipeline_result["decision_type"]=="escalated_manual_review"]["amount_rs"].sum()

print(f"Total failed payments in batch: Rs {total_failed_amount:,.2f} ({len(failed_in_test)} txns)")
print(f"Blocked as fraud by Guard: Rs {blocked_amount:,.2f} ({len(blocked_by_guard)} txns)")
print(f"  -> correctly fraud: {(blocked_by_guard['is_fraud']==1).sum()}, wrongly blocked genuine: {(blocked_by_guard['is_fraud']==0).sum()}")
print(f"Escalated to manual review: Rs {escalated_amount:,.2f}")
print(f"Auto-recovered: Rs {recovered_amount:,.2f}")

fraud_recovered = pipeline_result[(pipeline_result["actually_fraud"]==1) & (pipeline_result["recovered"]==True)]
print(f"\nRISK CHECK: fraud that slipped through AND got auto-recovered: {len(fraud_recovered)} txns, Rs {fraud_recovered['amount_rs'].sum():,.2f}")

Failed transactions in held-out test set: 600

Guard's verdict on these (predicted):
guard_predicts_fraud
False    540
True      60
Name: count, dtype: int64

Compare to REAL truth (only for OUR evaluation - the Helper never sees this):
Actually fraud      0   1
Guard says fraud         
False             538   2
True                0  60

Blocked by Guard: 60 txns, Rs 98,576.53
Sent to Helper: 540 txns, Rs 1,066,332.41

Wrongly blocked (actually genuine): 0
Fraud that slipped through to Helper: 2


========== FULL PIPELINE FINAL REPORT (held-out test batch) ==========
Total failed payments in batch: Rs 1,164,908.94 (600 txns)
Blocked as fraud by Guard: Rs 98,576.53 (60 txns)
  -> correctly fraud: 60, wrongly blocked genuine: 0
Escalated to manual review: Rs 310,283.57
Auto-recovered: Rs 298,950.77

RISK CHECK: fraud that slipped through AND got auto-recovered: 1 txns, Rs 733.16


In [17]:
# STEP 5 (v2): Professional multi-panel dashboard
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

np.random.seed(42)

# ---- Rebuild Guard + run the full pipeline (same as Step 4) ----
df = pd.read_csv("razorpay_style_transactions.csv")
df["device_customer_count"] = df.groupby("device_id")["customer_id"].transform("nunique")
method_dummies = pd.get_dummies(df["method"], prefix="method")
features = pd.concat([df[["amount", "international", "attempts_last_hour", "device_customer_count"]], method_dummies], axis=1)
target = df["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
guard_model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                             scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42)
guard_model.fit(X_train, y_train)

BEST_THRESHOLD = 0.70
test_probs = guard_model.predict_proba(X_test)[:, 1]
pipeline_df = df.loc[X_test.index].copy()
pipeline_df["guard_predicts_fraud"] = test_probs >= BEST_THRESHOLD
failed_in_test = pipeline_df[pipeline_df["status"] == "failed"].copy()
blocked_by_guard = failed_in_test[failed_in_test["guard_predicts_fraud"] == True]
sent_to_helper = failed_in_test[failed_in_test["guard_predicts_fraud"] == False]

recovery_action_map = {
    "insufficient_funds": "retry_later_24h", "card_declined": "send_reminder",
    "incorrect_otp": "send_reminder", "expired_card": "send_reminder",
    "bank_technical_error": "retry_immediately", "session_timed_out": "send_reminder",
    "insufficient_balance": "send_reminder", "wallet_service_down": "retry_later_2h",
    "payment_declined": "send_reminder", "upi_collect_expired": "resend_collect_request",
    "incorrect_pin": "send_reminder", "emi_not_supported": "suggest_alt_method",
    "issuer_unavailable": "retry_later_2h",
}
success_rate_map = {
    "retry_immediately": 0.65, "retry_later_2h": 0.55, "retry_later_24h": 0.45,
    "resend_collect_request": 0.50, "send_reminder": 0.30, "suggest_alt_method": 0.40,
}
MAX_ATTEMPTS = 4

log = []
for idx, row in sent_to_helper.iterrows():
    if row["attempts_last_hour"] >= MAX_ATTEMPTS:
        log.append({"amount_rs": row["amount"]/100, "action": "escalated", "recovered": False})
    else:
        action = recovery_action_map.get(row["error_reason"], "send_reminder")
        recovered = np.random.rand() < success_rate_map[action]
        log.append({"amount_rs": row["amount"]/100, "action": action, "recovered": recovered})

result = pd.DataFrame(log)

# ---- Compute all dashboard numbers ----
total_batch = failed_in_test["amount"].sum() / 100
recovered_amt = result[result["recovered"]]["amount_rs"].sum()
blocked_amt = blocked_by_guard["amount"].sum() / 100
escalated_amt = result[result["action"] == "escalated"]["amount_rs"].sum()
not_recovered_amt = total_batch - recovered_amt - blocked_amt - escalated_amt

precision = (blocked_by_guard["is_fraud"] == 1).sum() / len(blocked_by_guard) if len(blocked_by_guard) > 0 else 0
total_actual_fraud = (failed_in_test["is_fraud"] == 1).sum()
recall = (blocked_by_guard["is_fraud"] == 1).sum() / total_actual_fraud if total_actual_fraud > 0 else 0

by_action = result[result["recovered"]].groupby("action")["amount_rs"].sum().sort_values(ascending=False)
action_display_names = {
    "send_reminder": "Reminder sent", "retry_later_2h": "Retry (2h delay)",
    "retry_immediately": "Immediate retry", "resend_collect_request": "Resend UPI request",
    "retry_later_24h": "Retry (24h delay)", "suggest_alt_method": "Alt. method"
}
by_action.index = [action_display_names.get(a, a) for a in by_action.index]

# ---- Colors ----
INK = '#1a1a2e'; MUTED = '#6b7280'; CARD_BG = '#f8f9fb'; BORDER = '#e5e7eb'
BLUE = '#2563eb'; GREEN = '#16a34a'; AMBER = '#d97706'; RED = '#dc2626'

fig = plt.figure(figsize=(13, 9), facecolor='white')
gs = gridspec.GridSpec(12, 12, figure=fig, hspace=0.9, wspace=0.6)

fig.text(0.045, 0.965, 'RupeeGuard \u2014 Pipeline Performance Report', fontsize=19, fontweight='bold', color=INK)
fig.text(0.045, 0.938, f'Held-out test batch  \u00b7  {len(failed_in_test)} failed transactions  \u00b7  \u20b9{total_batch:,.0f} total value', fontsize=11, color=MUTED)

kpi_data = [
    ("TOTAL BATCH VALUE", f"\u20b9{total_batch/100000:.2f}L", f"{len(failed_in_test)} transactions", INK),
    ("AUTO-RECOVERED", f"\u20b9{recovered_amt/100000:.2f}L", f"{recovered_amt/total_batch*100:.1f}% of batch", GREEN),
    ("FRAUD BLOCKED", f"\u20b9{blocked_amt/100000:.2f}L", f"{len(blocked_by_guard)} txns \u00b7 {(blocked_by_guard['is_fraud']==0).sum()} false positives", RED),
    ("ESCALATED", f"\u20b9{escalated_amt/100000:.2f}L", f"{escalated_amt/total_batch*100:.1f}% \u00b7 manual review", AMBER),
]

kpi_ax = fig.add_axes([0.045, 0.76, 0.91, 0.13])
kpi_ax.axis('off'); kpi_ax.set_xlim(0, 4); kpi_ax.set_ylim(0, 1)
for i, (label, value, sub, color) in enumerate(kpi_data):
    box = FancyBboxPatch((i+0.03, 0.02), 0.94, 0.96, boxstyle="round,pad=0.02,rounding_size=0.04",
                          facecolor=CARD_BG, edgecolor=BORDER, linewidth=1)
    kpi_ax.add_patch(box)
    kpi_ax.text(i+0.5, 0.72, label, ha='center', va='center', fontsize=9.5, color=MUTED, fontweight='bold')
    kpi_ax.text(i+0.5, 0.42, value, ha='center', va='center', fontsize=22, color=color, fontweight='bold')
    kpi_ax.text(i+0.5, 0.15, sub, ha='center', va='center', fontsize=9, color=MUTED)

ax_donut = fig.add_subplot(gs[3:8, 0:5])
labels = ['Recovered', 'Fraud blocked', 'Escalated', 'Not recovered']
values = [recovered_amt, blocked_amt, escalated_amt, not_recovered_amt]
colors = [BLUE, RED, AMBER, '#cbd5e1']
ax_donut.pie(values, colors=colors, startangle=90, wedgeprops=dict(width=0.38, edgecolor='white', linewidth=3))
ax_donut.text(0, 0.08, f'\u20b9{total_batch/100000:.2f}L', ha='center', va='center', fontsize=17, fontweight='bold', color=INK)
ax_donut.text(0, -0.13, 'total batch', ha='center', va='center', fontsize=9.5, color=MUTED)
ax_donut.set_title('Outcome breakdown', fontsize=12.5, fontweight='bold', color=INK, loc='left', x=-0.55, y=1.05)
legend_elements = [mpatches.Patch(facecolor=colors[i], label=f'{labels[i]}  {values[i]/sum(values)*100:.1f}%') for i in range(4)]
ax_donut.legend(handles=legend_elements, loc='center', bbox_to_anchor=(0.5, -0.22), ncol=2,
                 frameon=False, fontsize=9.5, handlelength=1, handleheight=1, columnspacing=1.2)

ax_metrics = fig.add_subplot(gs[3:6, 6:12])
ax_metrics.axis('off')
ax_metrics.set_title('Fraud detection model performance', fontsize=12.5, fontweight='bold', color=INK, loc='left', x=0)
metrics = [('Precision', precision), ('Recall', recall)]
for (name, val), y in zip(metrics, [0.62, 0.15]):
    ax_metrics.text(0, y+0.22, name, fontsize=10.5, color=MUTED, fontweight='bold')
    ax_metrics.text(1.0, y+0.22, f'{val*100:.1f}%', fontsize=10.5, color=INK, fontweight='bold', ha='right')
    ax_metrics.add_patch(FancyBboxPatch((0, y), 1.0, 0.12, boxstyle="round,pad=0,rounding_size=0.06", facecolor='#e5e7eb', edgecolor='none'))
    ax_metrics.add_patch(FancyBboxPatch((0, y), val, 0.12, boxstyle="round,pad=0,rounding_size=0.06", facecolor=BLUE, edgecolor='none'))
ax_metrics.set_xlim(0, 1.02); ax_metrics.set_ylim(0, 1.0)
ax_metrics.text(0, -0.08, f'{(blocked_by_guard["is_fraud"]==1).sum()} of {total_actual_fraud} fraud cases caught  \u00b7  {(blocked_by_guard["is_fraud"]==0).sum()} genuine customers wrongly blocked', fontsize=9, color=MUTED)

ax_bar = fig.add_subplot(gs[7:11, 6:12])
amounts_L = (by_action / 100000).values
y_pos = np.arange(len(by_action))
ax_bar.barh(y_pos, amounts_L, color=BLUE, height=0.55, zorder=3)
ax_bar.set_yticks(y_pos); ax_bar.set_yticklabels(by_action.index, fontsize=9.5, color=INK)
ax_bar.invert_yaxis()
ax_bar.set_xlabel('Amount recovered (\u20b9 lakhs)', fontsize=9.5, color=MUTED)
ax_bar.set_title('Recovered value by action type', fontsize=12.5, fontweight='bold', color=INK, loc='left', x=0, y=1.08)
for spine in ['top', 'right', 'left']:
    ax_bar.spines[spine].set_visible(False)
ax_bar.spines['bottom'].set_color(BORDER)
ax_bar.tick_params(left=False, labelsize=9, colors=MUTED)
ax_bar.grid(axis='x', color=BORDER, linewidth=0.7, zorder=0)
ax_bar.set_axisbelow(True)
for i, v in enumerate(by_action.values):
    ax_bar.text(amounts_L[i] + 0.02, i, f'\u20b9{v/1000:.0f}K', va='center', fontsize=8.5, color=MUTED)

fig.text(0.045, 0.025, 'Evaluated on a single held-out split (test data unseen during model training)  \u00b7  Fraud threshold selected via cost-minimization, not default 0.5', fontsize=8.5, color=MUTED)

plt.savefig('final_report_chart.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: final_report_chart.png")

Saved: final_report_chart.png


In [18]:
# DEMO: Run a single example transaction through RupeeGuard end-to-end
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

np.random.seed(42)

# ---- Rebuild the Guard ----
df = pd.read_csv("razorpay_style_transactions.csv")
df["device_customer_count"] = df.groupby("device_id")["customer_id"].transform("nunique")
method_dummies_ref = pd.get_dummies(df["method"], prefix="method")
FEATURE_COLUMNS = ["amount", "international", "attempts_last_hour", "device_customer_count"] + list(method_dummies_ref.columns)

features = pd.concat([df[["amount", "international", "attempts_last_hour", "device_customer_count"]], method_dummies_ref], axis=1)
target = df["is_fraud"]
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
guard_model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                             scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42)
guard_model.fit(X_train, y_train)

GUARD_THRESHOLD = 0.70

recovery_action_map = {
    "insufficient_funds": "retry_later_24h", "card_declined": "send_reminder",
    "incorrect_otp": "send_reminder", "expired_card": "send_reminder",
    "bank_technical_error": "retry_immediately", "session_timed_out": "send_reminder",
    "insufficient_balance": "send_reminder", "wallet_service_down": "retry_later_2h",
    "payment_declined": "send_reminder", "upi_collect_expired": "resend_collect_request",
    "incorrect_pin": "send_reminder", "emi_not_supported": "suggest_alt_method",
    "issuer_unavailable": "retry_later_2h",
}
MAX_ATTEMPTS_BEFORE_ESCALATION = 4

def process_transaction(txn):
    """Takes ONE transaction (a dict) and runs it through the full RupeeGuard pipeline."""
    row = {
        "amount": txn["amount"], "international": txn["international"],
        "attempts_last_hour": txn["attempts_last_hour"], "device_customer_count": txn["device_customer_count"],
    }
    for col in method_dummies_ref.columns:
        row[col] = (col == f"method_{txn['method']}")
    X_new = pd.DataFrame([row])[FEATURE_COLUMNS]

    fraud_prob = guard_model.predict_proba(X_new)[0, 1]

    if fraud_prob >= GUARD_THRESHOLD:
        return {
            "guard_fraud_probability": round(float(fraud_prob), 3),
            "decision": "BLOCKED", "recovery_action": None,
            "reasoning": f"Guard flagged this as fraud with {fraud_prob*100:.1f}% confidence. No recovery attempted."
        }

    if txn["attempts_last_hour"] >= MAX_ATTEMPTS_BEFORE_ESCALATION:
        return {
            "guard_fraud_probability": round(float(fraud_prob), 3),
            "decision": "ESCALATED_TO_HUMAN", "recovery_action": "escalate_to_human",
            "reasoning": f"Cleared as genuine, but already failed {txn['attempts_last_hour']} times. Stopping rule triggered."
        }

    action = recovery_action_map.get(txn["error_reason"], "send_reminder")
    return {
        "guard_fraud_probability": round(float(fraud_prob), 3),
        "decision": "RECOVERY_ATTEMPTED", "recovery_action": action,
        "reasoning": f"Cleared as genuine. Failure reason '{txn['error_reason']}' mapped to action '{action}'."
    }


# ===== EXAMPLE INPUT 1: a genuine failed payment =====
example_genuine = {
    "amount": 149900,             # Rs 1,499.00 (amount in paise)
    "method": "upi",
    "international": False,
    "attempts_last_hour": 1,
    "device_customer_count": 1,   # only ever used by 1 customer - looks normal
    "error_reason": "insufficient_funds",
}

print("INPUT: ", example_genuine)
print("OUTPUT:", process_transaction(example_genuine))


# ===== EXAMPLE INPUT 2: a fraud-looking payment =====
example_fraud = {
    "amount": 850000,             # Rs 8,500.00
    "method": "card",
    "international": True,
    "attempts_last_hour": 18,     # way too many attempts, fast
    "device_customer_count": 24,  # shared by 24 different "customers" - classic fraud ring
    "error_reason": "card_declined",
}

print("\nINPUT: ", example_fraud)
print("OUTPUT:", process_transaction(example_fraud))

INPUT:  {'amount': 149900, 'method': 'upi', 'international': False, 'attempts_last_hour': 1, 'device_customer_count': 1, 'error_reason': 'insufficient_funds'}
OUTPUT: {'guard_fraud_probability': 0.0, 'decision': 'RECOVERY_ATTEMPTED', 'recovery_action': 'retry_later_24h', 'reasoning': "Cleared as genuine. Failure reason 'insufficient_funds' mapped to action 'retry_later_24h'."}

INPUT:  {'amount': 850000, 'method': 'card', 'international': True, 'attempts_last_hour': 18, 'device_customer_count': 24, 'error_reason': 'card_declined'}
OUTPUT: {'guard_fraud_probability': 1.0, 'decision': 'BLOCKED', 'recovery_action': None, 'reasoning': 'Guard flagged this as fraud with 100.0% confidence. No recovery attempted.'}
